# Module 05 — Lesson 06
# Handling Missing Values

---

> `tips.csv` from the last three lessons was unusually well-behaved -- every cell had a value. That's rare. Sensors fail, forms get skipped, surveys go unanswered -- almost every real dataset has gaps. This lesson is about the two questions those gaps force on you: *how much is missing*, and *what do I do about it*.

For this lesson we switch to `planets.csv` -- a real dataset of confirmed exoplanet discoveries (1,035 rows) that genuinely has missing values in three of its columns, unlike `tips.csv`.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Detect missing values with `.isna()` / `.isnull()`, and count them per column with `.sum()`
- Judge severity with the *percentage* missing per column, not just the raw count
- Remove missing data with `.dropna()` — scoped by `subset`, `thresh`, or `axis`
- Fill missing data with `.fillna()` — a constant, the column mean/median, or forward/backward fill
- Recognize that missing values aren't always `NaN` — some arrive disguised as strings like `"N/A"`

In [1]:
import pandas as pd

planets = pd.read_csv("data/planets.csv")
print(f"planets.shape: {planets.shape}")
print(planets.head())

planets.shape: (1035, 6)
            method  number  orbital_period   mass  distance  year
0  Radial Velocity       1         269.300   7.10     77.40  2006
1  Radial Velocity       1         874.774   2.21     56.95  2008
2  Radial Velocity       1         763.000   2.60     19.84  2011
3  Radial Velocity       1         326.030  19.40    110.62  2007
4  Radial Velocity       1         516.220  10.50    119.47  2009


---
## 1. Detecting Missing Values

`.isna()` (identical to `.isnull()`) returns a same-shaped DataFrame of `True`/`False`. Chain `.sum()` to collapse that down to a per-column count -- your first real diagnostic on any new dataset.

In [2]:
missing_counts = planets.isna().sum()
print("Missing values per column:")
print(missing_counts)
print()
print(f"Total missing cells: {planets.isna().sum().sum()}")

Missing values per column:
method              0
number              0
orbital_period     43
mass              522
distance          227
year                0
dtype: int64

Total missing cells: 792


`.info()` shows the same story a different way -- compare each column's "Non-Null Count" against the total row count.

In [3]:
planets.info()

<class 'pandas.DataFrame'>
RangeIndex: 1035 entries, 0 to 1034
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   method          1035 non-null   str    
 1   number          1035 non-null   int64  
 2   orbital_period  992 non-null    float64
 3   mass            513 non-null    float64
 4   distance        808 non-null    float64
 5   year            1035 non-null   int64  
dtypes: float64(3), int64(2), str(1)
memory usage: 48.6 KB


---
## 2. Judging Severity: Percentage, Not Just Count

43 missing values sounds like a lot -- until you see it's only 4% of 1,035 rows. `mass` missing 522 values, on the other hand, is over half the column. The right strategy depends entirely on this percentage.

In [4]:
missing_pct = (planets.isna().mean() * 100).round(1)
print("Percentage missing per column:")
print(missing_pct)

Percentage missing per column:
method             0.0
number             0.0
orbital_period     4.2
mass              50.4
distance          21.9
year               0.0
dtype: float64


---
## 3. Dropping Missing Values: `.dropna()`

By default, `.dropna()` drops any **row** that has a `NaN` in **any** column -- often far more aggressive than you want. Scope it down with `subset` (only care about certain columns), `thresh` (keep rows with at least N non-null values), or `axis=1` (drop columns instead of rows).

In [5]:
# Default: drop a row if ANY column has a missing value
dropped_any = planets.dropna()
print(f"Default dropna(): {planets.shape[0]} -> {dropped_any.shape[0]} rows "
      f"({planets.shape[0] - dropped_any.shape[0]} dropped)")

Default dropna(): 1035 -> 498 rows (537 dropped)


In [6]:
# subset: only drop a row if a SPECIFIC column is missing
dropped_distance_only = planets.dropna(subset=["distance"])
print(f"dropna(subset=['distance']): {planets.shape[0]} -> {dropped_distance_only.shape[0]} rows")

dropna(subset=['distance']): 1035 -> 808 rows


In [7]:
# thresh: keep a row if it has AT LEAST this many non-null values (out of 6 columns)
dropped_thresh = planets.dropna(thresh=5)
print(f"dropna(thresh=5): {planets.shape[0]} -> {dropped_thresh.shape[0]} rows")

dropna(thresh=5): 1035 -> 791 rows


In [8]:
# axis=1: drop whole COLUMNS instead of rows -- here, any column with at least one NaN
dropped_columns = planets.dropna(axis=1)
print(f"dropna(axis=1) columns kept: {list(dropped_columns.columns)}")

dropna(axis=1) columns kept: ['method', 'number', 'year']


---
## 4. Filling Missing Values: `.fillna()`

Instead of losing rows, `.fillna()` replaces `NaN` with something else: a constant, a computed statistic, or a neighboring value.

In [9]:
# Fill with a constant
orbital_filled_zero = planets["orbital_period"].fillna(0)
print(f"Missing after fillna(0): {orbital_filled_zero.isna().sum()}")
print()

# Fill with the column's median -- usually a safer default than a constant
orbital_median = planets["orbital_period"].median()
orbital_filled_median = planets["orbital_period"].fillna(orbital_median)
print(f"orbital_period median: {orbital_median:.2f}")
print(f"Missing after fillna(median): {orbital_filled_median.isna().sum()}")

Missing after fillna(0): 0

orbital_period median: 39.98
Missing after fillna(median): 0


In [10]:
# ffill / bfill: carry the previous (or next) valid value forward/backward --
# useful for ordered data like a time series, less so for this dataset
small_example = pd.Series([10, None, None, 40, None])
print("Original:", small_example.tolist())
print("ffill:   ", small_example.ffill().tolist())
print("bfill:   ", small_example.bfill().tolist())

Original: [10.0, nan, nan, 40.0, nan]
ffill:    [10.0, 10.0, 10.0, 40.0, 40.0]
bfill:    [10.0, 40.0, 40.0, 40.0, nan]


---
## 5. Choosing a Strategy, Column by Column

There's no single right answer -- it depends on how much is missing and what the column means:

- **`orbital_period`** (4% missing) -- safe to fill with the median, or drop those few rows
- **`distance`** (22% missing) -- borderline; filling with median is reasonable if you need every row, dropping is reasonable if you don't
- **`mass`** (over 50% missing) -- too much to safely fill; better to drop the column entirely if it's not central to your analysis, or keep the NaNs and exclude the column from mass-specific calculations

In [11]:
planets_clean = planets.copy()
planets_clean["orbital_period"] = planets_clean["orbital_period"].fillna(planets_clean["orbital_period"].median())
planets_clean["distance"] = planets_clean["distance"].fillna(planets_clean["distance"].median())
planets_clean = planets_clean.drop(columns=["mass"])

print("After applying our per-column strategy:")
print(planets_clean.isna().sum())

After applying our per-column strategy:
method            0
number            0
orbital_period    0
distance          0
year              0
dtype: int64


---
## ⚠️ Common Mistakes

In [ ]:
# -- Mistake 1: Calling dropna() with no arguments on the whole DataFrame -------

print("Mistake 1 — default dropna() drops a row if ANY column is missing, even")
print("one you don't care about:")
over_dropped = planets.dropna()
print(f"  planets.dropna() keeps only {over_dropped.shape[0]} of {planets.shape[0]} rows")
print("  because 'mass' alone is missing in over half the dataset.")
print()
print("  Correct: scope it to the column(s) that actually matter:")
scoped = planets.dropna(subset=["distance"])
print(f"  planets.dropna(subset=['distance']) keeps {scoped.shape[0]} rows")

In [ ]:
# -- Mistake 2: Filling with the mean when the column is skewed by outliers -----

print("Mistake 2 — the mean is pulled around by outliers; the median often isn't:")
print(f"  orbital_period mean:   {planets['orbital_period'].mean():.1f}")
print(f"  orbital_period median: {planets['orbital_period'].median():.1f}")
print("  That's a huge gap -- a few extremely long orbital periods drag the mean")
print("  far above where most of the data actually sits.")
print()
print("  Correct: prefer the median for skewed numeric columns, unless you've")
print("  confirmed the column is roughly symmetric.")

In [ ]:
# -- Mistake 3: Missing values disguised as strings pandas doesn't recognize ----

import io

disguised = """planet;distance
Kepler-10b;N/A
Kepler-11b;60.4
Kepler-12b;--
"""

print("Mistake 3 — pandas auto-recognizes SOME placeholder strings (like 'N/A'),")
print("but NOT arbitrary ones your data source happens to use (like '--'):")
naive = pd.read_csv(io.StringIO(disguised), sep=";")
print(naive)
print(f"  isna().sum() sees: {naive['distance'].isna().sum()} missing "
      f"('N/A' was caught automatically, '--' was not -- should be 2)")
print()

print("  Correct: never assume -- always add your own na_values= for anything")
print("  project-specific, on top of whatever pandas already catches:")
fixed = pd.read_csv(io.StringIO(disguised), sep=";", na_values=["--"])
print(fixed)
print(f"  isna().sum() now sees: {fixed['distance'].isna().sum()} missing (correct)")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Count and Percentage

Load `data/planets.csv`. Print the number of missing values per column, and the percentage missing per column (rounded to 1 decimal place).

In [ ]:
# Your code here

### Exercise 2 — Scoped Dropping

Drop rows from `planets` only where `"orbital_period"` is missing (use `subset`). Print the shape before and after.

In [ ]:
# Your code here

### Exercise 3 — Fill with Median

Fill the missing values in `"distance"` with that column's median. Confirm with `.isna().sum()` that no missing values remain in that column.

In [ ]:
# Your code here

### Exercise 4 — Drop a Column

Drop the `"mass"` column entirely from `planets` (it's missing over half its values). Print the remaining column names.

In [ ]:
# Your code here

### Exercise 5 — (Challenge) Disguised Missing Values

This tiny CSV uses `"unknown"` for missing data instead of leaving cells blank:

```python
raw = """star;year
Star A;2001
Star B;unknown
Star C;2015
"""
```

Read it with `pd.read_csv(io.StringIO(raw))` two ways: once normally, and once passing `na_values=["unknown"]`. Print `.isna().sum()` for `"year"` both times to show the difference.

In [ ]:
# Your code here

---
## 📌 Key Takeaways

- **`.isna().sum()` (counts) and `.isna().mean()` (percentages) are the first checks on any real dataset** — the right strategy for a column missing 4% of its values is very different from one missing 50%.

- **`.dropna()` and `.fillna()` default to touching every column** — scope them with `subset`, `thresh`, `axis`, or by selecting a single column first, or you'll lose (or alter) far more data than you meant to.

- **Pandas auto-recognizes common missing-value markers (like `"N/A"`, `"NULL"`) but not project-specific ones** — placeholders like `"--"` or `"unknown"` silently pass through as regular text unless you add them yourself with `na_values=` at read time (or `.replace()` afterward).

---
## What's Next?

**Lesson 07 — GroupBy** takes a cleaned dataset and starts summarizing it — "average orbital period by discovery method," "count of planets discovered per year."